# Data Cleaning

Raw data: [Kaggle KKBox churn data](https://www.kaggle.com/competitions/kkbox-churn-prediction-challenge/data).

**Observation period:** 2015-01-01 – 2017-02-28

**Eligible users**
1. Initial account registration occurred within the observation window.
2. At least one transaction.

**Then** random sample 1,000 users and extract the corresponding data from the raw datasets.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from kkbox_subsample import KKBoxSubsampleConfig, run_pipeline

In [2]:
DATA_DIR = Path("kkbox-churn-prediction-challenge")

cfg = KKBoxSubsampleConfig(
    raw_dir=DATA_DIR,
    out_dir=Path("kkbox-subsample"),
    obs_start=20150101,
    obs_end=20170228,
    n_sample=5000,
    random_state=641,
)

sample_users = run_pipeline(cfg)


Scanning registrations: kkbox-churn-prediction-challenge/members_v3.csv
  Registered in observation window: 4,194,804
Scanning first transaction dates (2 file(s))...
  With at least one transaction: 900,214
  With registration + transaction: 900,214
Saved eligible cohort -> kkbox-subsample/eligible_users.csv
Sampling 5000 users...
Saved sample metadata -> kkbox-subsample/sample_users.csv
Saved sample ids -> kkbox-subsample/sample_msno.csv
Extracting subsample tables (chunked)...
  Extracting members...
    -> kkbox-subsample/subsample/members.csv (5,000 rows)
  Extracting transactions...
    -> kkbox-subsample/subsample/transactions.csv (31,731 rows)
  Extracting user_logs...
    -> kkbox-subsample/subsample/user_logs.csv (525,835 rows)
  Extracting transactions_v2...
    -> kkbox-subsample/subsample/transactions_v2.csv (2,788 rows)
  Extracting user_logs_v2...
    -> kkbox-subsample/subsample/user_logs_v2.csv (40,669 rows)
  members: 5,000 rows
  transactions: 31,731 rows
  user_logs:

## Build panel data (user × month since joining)

**Periods:** `user_tenure` = 0, 1, 2, … **30-day blocks** since `first_transaction_date` (tenure 0 = days 0–29, tenure 1 = days 30–59, …), matching subscription `payment_plan_days`.

**Subscription status** (each user-month):

| Status | Meaning |
|--------|---------|
| `active` | Paid membership covers at least part of the month. |
| `paused` | Lapsed after a spell (no renewal within 30 days of expiry) but **resubscribes** later in the observation window. |
| `churn` | First lapse with **no** return before 2017-02-28; absorbing (panel ends that month). |

**Other fields:** `cum_hours`, `lagged_cum_hours`, `price` (per 30-day period; multi-month plans scaled by `payment_plan_days / 30`), `auto_renewal`.

In [3]:
SUBSAMPLE_DIR = Path("kkbox-subsample")
OBS_END = pd.Timestamp("2017-02-28")
RENEWAL_GRACE_DAYS = 30
DAYS_PER_PERIOD = 30

users = pd.read_csv(SUBSAMPLE_DIR / "sample_users.csv", dtype={"msno": str})
members = pd.read_csv(SUBSAMPLE_DIR / "subsample/members.csv", dtype={"msno": str})
transactions = pd.read_csv(SUBSAMPLE_DIR / "subsample/transactions_merged.csv", dtype={"msno": str})
logs = pd.read_csv(SUBSAMPLE_DIR / "subsample/user_logs_daily.csv", dtype={"msno": str})

# Attach sampled user profile fields from members.csv
users = users.merge(
    members[["msno", "city", "bd", "gender", "registered_via"]],
    on="msno",
    how="left",
)

users["join_date"] = pd.to_datetime(users["first_transaction_date"].astype(str), format="%Y%m%d")
transactions["txn_date"] = pd.to_datetime(transactions["transaction_date"].astype(int).astype(str), format="%Y%m%d")
transactions["expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(int).astype(str), format="%Y%m%d"
)
logs["log_date"] = pd.to_datetime(logs["date"].astype(int).astype(str), format="%Y%m%d")

In [4]:
def months_since_join(join_date: pd.Series | pd.Timestamp, event_date: pd.Series | pd.Timestamp) -> pd.Series:
    """Integer 30-day periods since join (0 = days 0–29 after first_transaction_date)."""
    join_s = join_date if isinstance(join_date, pd.Series) else pd.Series([join_date])
    event_s = event_date if isinstance(event_date, pd.Series) else pd.Series([event_date])
    days = (event_s - join_s).dt.days
    return (days // DAYS_PER_PERIOD).clip(lower=0)


def tenure_period_bounds(join_date: pd.Timestamp, tenure: int) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Inclusive date range for user_tenure t: join + 30*t through join + 30*(t+1) - 1 day."""
    start = join_date + pd.Timedelta(days=DAYS_PER_PERIOD * tenure)
    end = join_date + pd.Timedelta(days=DAYS_PER_PERIOD * (tenure + 1)) - pd.Timedelta(days=1)
    return start, end


def to_monthly_price(plan_price: pd.Series, plan_days: pd.Series) -> pd.Series:
    """Map plan_list_price to price per 30-day renewal period (structural model month)."""
    n_periods = (plan_days / DAYS_PER_PERIOD).clip(lower=1)
    return np.where(plan_days > DAYS_PER_PERIOD, plan_price / n_periods, plan_price)


def _period_overlaps(period_start: pd.Timestamp, period_end: pd.Timestamp, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    return period_start <= end and period_end >= start


def _renewed_within_grace(spells: pd.DataFrame, expire: pd.Timestamp, lapse_confirm: pd.Timestamp) -> bool:
    return (
        (spells["txn_date"] >= expire)
        & (spells["txn_date"] <= lapse_confirm)
        & (spells["expire_date"] > expire)
    ).any()


def _still_covered_after_expiry(spells: pd.DataFrame, expire: pd.Timestamp) -> bool:
    """Another paid spell still covers the day after this row's expiry (e.g. 90-day plan)."""
    day_after = expire + pd.Timedelta(days=1)
    return ((spells["txn_date"] <= day_after) & (spells["expire_date"] >= day_after)).any()


def build_monthly_subscription_status(
    txn: pd.DataFrame, users_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return (monthly_status, user_panel_bounds).

    active: paid coverage overlaps the 30-day tenure period.
    paused: lapsed (no renewal within 30d) but resubscribes before OBS_END.
    churn: first lapse with no return; panel ends that month (absorbing).
    """
    txn = txn.merge(users_df[["msno", "join_date"]], on="msno")
    status_rows: list[dict] = []
    bounds_rows: list[dict] = []

    for msno, grp in txn.groupby("msno"):
        join_date = grp["join_date"].iloc[0]
        spells = grp[grp["is_cancel"] == 0].sort_values(["txn_date", "transaction_date"])
        intervals: list[tuple[pd.Timestamp, pd.Timestamp, str]] = []
        terminal_churn_tenure: int | None = None

        for row in spells.itertuples(index=False):
            intervals.append((row.txn_date, row.expire_date, "active"))

        for row in spells.itertuples(index=False):
            expire = row.expire_date
            if expire > OBS_END:
                continue
            lapse_confirm = min(expire + pd.Timedelta(days=RENEWAL_GRACE_DAYS), OBS_END)
            if _renewed_within_grace(spells, expire, lapse_confirm) or _still_covered_after_expiry(
                spells, expire
            ):
                continue

            later = spells[spells["txn_date"] > lapse_confirm]
            if len(later) and later.iloc[0]["txn_date"] <= OBS_END:
                pause_start = lapse_confirm + pd.Timedelta(days=1)
                pause_end = later.iloc[0]["txn_date"] - pd.Timedelta(days=1)
                if pause_start <= pause_end:
                    intervals.append((pause_start, pause_end, "paused"))
            else:
                churn_start = lapse_confirm + pd.Timedelta(days=1)
                if churn_start <= OBS_END:
                    intervals.append((churn_start, OBS_END, "churn"))
                terminal_churn_tenure = int(months_since_join(join_date, lapse_confirm).iloc[0])
                break

        max_tenure = int(months_since_join(join_date, OBS_END).iloc[0])
        panel_end = terminal_churn_tenure if terminal_churn_tenure is not None else max_tenure
        bounds_rows.append(
            {
                "msno": msno,
                "panel_end": panel_end,
                "first_churn_month": terminal_churn_tenure,
            }
        )

        _STATUS_RANK = {"active": 3, "churn": 2, "paused": 1}
        _RANK_TO_STATUS = {3: "active", 2: "churn", 1: "paused"}

        for tenure in range(panel_end + 1):
            period_start, period_end = tenure_period_bounds(join_date, tenure)
            best_rank = 0
            for start, end, label in intervals:
                if _period_overlaps(period_start, period_end, start, end):
                    best_rank = max(best_rank, _STATUS_RANK[label])
            status_rows.append(
                {
                    "msno": msno,
                    "user_tenure": tenure,
                    "subscription_status": _RANK_TO_STATUS.get(best_rank, "active"),
                }
            )

    return pd.DataFrame(status_rows), pd.DataFrame(bounds_rows)


monthly_status, user_panel_bounds = build_monthly_subscription_status(transactions, users)
users = users.merge(user_panel_bounds, on="msno", how="left")

print(monthly_status["subscription_status"].value_counts())
print(f"Users with terminal churn: {users['first_churn_month'].notna().sum():,}")
print(f"Users with at least one paused month: {monthly_status.loc[monthly_status.subscription_status == 'paused', 'msno'].nunique():,}")
monthly_status.head(10)

subscription_status
active    37769
churn      2196
paused     1979
Name: count, dtype: int64
Users with terminal churn: 2,429
Users with at least one paused month: 465


,msno,user_tenure,subscription_status
0,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,0,active
1,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,1,active
2,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,2,active
3,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,3,active
4,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,4,active
5,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,5,active
6,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,6,active
7,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,7,active
8,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,8,active
9,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,9,active


In [5]:
# --- monthly listening hours and cumulative hours ---

logs = logs.merge(users[["msno", "join_date"]], on="msno")
logs["user_tenure"] = months_since_join(logs["join_date"], logs["log_date"])
logs = logs[(logs["user_tenure"] >= 0) & (logs["log_date"] <= OBS_END)]

monthly_hours = (
    logs.groupby(["msno", "user_tenure"], as_index=False)["hours"].sum()
    .rename(columns={"hours": "monthly_hours"})
)
monthly_hours = monthly_hours.sort_values(["msno", "user_tenure"])
monthly_hours["cum_hours"] = monthly_hours.groupby("msno")["monthly_hours"].cumsum()

monthly_hours.head(10)

,msno,user_tenure,monthly_hours,cum_hours
0,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,0,20.785846,20.785846
1,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,1,64.728187,85.514034
2,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,2,83.284939,168.798972
3,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,3,39.617036,208.416008
4,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,4,55.922110,264.338118
5,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,5,111.479809,375.817927
6,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,6,89.104756,464.922683
7,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,7,53.534884,518.457567
8,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,8,65.377426,583.834993
9,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,9,56.251836,640.086829


In [6]:
# --- monthly subscription price and auto-renew (from renewals; forward-filled) ---

renewals = transactions[transactions["is_cancel"] == 0].copy()
renewals = renewals.merge(users[["msno", "join_date"]], on="msno")
renewals["user_tenure"] = months_since_join(renewals["join_date"], renewals["txn_date"])
renewals = renewals[renewals["user_tenure"] >= 0]

# If multiple renewals in a period, keep the last by transaction date
monthly_sub = (
    renewals.sort_values(["msno", "user_tenure", "transaction_date"])
    .groupby(["msno", "user_tenure"], as_index=False)
    .tail(1)[
        [
            "msno",
            "user_tenure",
            "plan_list_price",
            "payment_plan_days",
            "is_auto_renew",
            "transaction_date",
            "membership_expire_date",
        ]
    ]
)
monthly_sub["price"] = to_monthly_price(
    monthly_sub["plan_list_price"], monthly_sub["payment_plan_days"]
)
monthly_sub = monthly_sub.rename(columns={"is_auto_renew": "auto_renewal"}).drop(
    columns=["plan_list_price", "payment_plan_days"]
)

monthly_sub.head(10)

,msno,user_tenure,auto_renewal,transaction_date,membership_expire_date,price
1,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,0,1,20160131,20160229,149.0
2,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,1,1,20160229,20160331,149.0
3,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,2,1,20160331,20160430,149.0
4,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,3,1,20160430,20160531,149.0
5,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,4,1,20160531,20160630,149.0
6,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,5,1,20160630,20160731,149.0
7,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,6,1,20160731,20160831,149.0
8,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,7,1,20160831,20160930,149.0
9,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,8,1,20160930,20161031,149.0
10,++92FghbCQPqDmQ96QzNiuEMoDxrMOmuaisu1UCrYn0=,9,1,20161031,20161130,149.0


In [7]:
# --- build user-month panel (truncate at terminal churn) ---

panel = monthly_status.merge(monthly_hours, on=["msno", "user_tenure"], how="left")
panel = panel.merge(monthly_sub, on=["msno", "user_tenure"], how="left")

panel = panel.sort_values(["msno", "user_tenure"])
for col in ("price", "auto_renewal"):
    panel[col] = panel.groupby("msno")[col].ffill()

panel["monthly_hours"] = panel["monthly_hours"].fillna(0.0)
panel["cum_hours"] = panel.groupby("msno")["monthly_hours"].cumsum()
panel["cum_hours"] = panel.groupby("msno")["cum_hours"].shift(1).fillna(0.0)

# Drop users with zero active price (free trials / plan_list_price data errors)
zero_price_active_msno = panel.loc[
    (panel["subscription_status"] == "active") & (panel["price"] <= 0), "msno"
].unique()

# Drop users with any active period that has no overlapping paid coverage
paid_spells = transactions.loc[transactions["is_cancel"] == 0, ["msno", "txn_date", "expire_date"]].copy()
active_rows = panel.loc[panel["subscription_status"] == "active", ["msno", "user_tenure"]]
active_rows = active_rows.merge(users[["msno", "join_date"]], on="msno", how="left")

def has_paid_overlap(row: pd.Series) -> bool:
    period_start, period_end = tenure_period_bounds(row["join_date"], int(row["user_tenure"]))
    spells = paid_spells.loc[paid_spells["msno"] == row["msno"]]
    return ((spells["txn_date"] <= period_end) & (spells["expire_date"] >= period_start)).any()

active_no_coverage = active_rows.loc[~active_rows.apply(has_paid_overlap, axis=1), "msno"].unique()

# Drop users with listening during paused months (likely transaction/status mismatch)
paused_with_hours = (
    panel.loc[panel["subscription_status"] == "paused"]
    .groupby("msno")["monthly_hours"]
    .apply(lambda s: (s > 0).any())
)
paused_with_hours_msno = paused_with_hours[paused_with_hours].index

# New filter 1: drop users with any plan longer than 30 days
long_plan_msno = transactions.loc[
    (transactions["is_cancel"] == 0) & (transactions["payment_plan_days"] > DAYS_PER_PERIOD),
    "msno",
].unique()

# New filter 2: require constant active-period price and auto_renewal within user
active_panel = panel.loc[panel["subscription_status"] == "active", ["msno", "price", "auto_renewal"]].copy()
nonconstant_price_msno = active_panel.groupby("msno")["price"].nunique(dropna=True)
nonconstant_price_msno = nonconstant_price_msno[nonconstant_price_msno > 1].index

nonconstant_autorenew_msno = active_panel.groupby("msno")["auto_renewal"].nunique(dropna=True)
nonconstant_autorenew_msno = nonconstant_autorenew_msno[nonconstant_autorenew_msno > 1].index

# Drop users with invalid cum_hours (negative / missing; breaks hour-bin discretization)
invalid_cum_hours_msno = panel.loc[~panel["cum_hours"].ge(0), "msno"].unique()

drop_msno = (
    pd.Index(active_no_coverage)
    .union(paused_with_hours_msno)
    .union(zero_price_active_msno)
    .union(long_plan_msno)
    .union(nonconstant_price_msno)
    .union(nonconstant_autorenew_msno)
    .union(invalid_cum_hours_msno)
)
n_before = panel["msno"].nunique()
panel = panel[~panel["msno"].isin(drop_msno)].copy()

print(
    f"Dropped {len(zero_price_active_msno)} users with active price <= 0; "
    f"{len(active_no_coverage)} with active months lacking paid overlap; "
    f"{len(paused_with_hours_msno)} with hours > 0 during paused months; "
    f"{len(long_plan_msno)} with plan > 30 days; "
    f"{len(nonconstant_price_msno)} with non-constant active price; "
    f"{len(nonconstant_autorenew_msno)} with non-constant active auto_renewal; "
    f"{len(invalid_cum_hours_msno)} with invalid cum_hours "
    f"({n_before} -> {panel['msno'].nunique()} users)"
)

panel.head(12)

Dropped 1542 users with active price <= 0; 106 with active months lacking paid overlap; 175 with hours > 0 during paused months; 621 with plan > 30 days; 832 with non-constant active price; 222 with non-constant active auto_renewal; 20 with invalid cum_hours (5000 -> 2465 users)


,msno,user_tenure,subscription_status,monthly_hours,cum_hours,auto_renewal,transaction_date,membership_expire_date,price
29,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,0,active,39.826280,0.000000,0.0,20150215.0,20150317.0,149.0
30,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,1,active,16.865030,39.826280,0.0,20150320.0,20150419.0,149.0
31,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,2,active,20.236815,56.691311,0.0,20150421.0,20150521.0,149.0
32,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,3,active,17.657869,76.928126,0.0,20150522.0,20150621.0,149.0
33,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,4,active,11.805867,94.585995,0.0,20150621.0,20150721.0,149.0
34,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,5,active,2.028575,106.391862,0.0,NaN,NaN,149.0
35,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,6,paused,0.000000,108.420437,0.0,NaN,NaN,149.0
36,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,7,active,13.498009,108.420437,0.0,20150919.0,20151019.0,149.0
37,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,8,active,0.971087,121.918446,0.0,NaN,NaN,149.0
38,++nqzXpLn1fFKdCWEO5X5ufg5RJV9FgPMsKuHrnwk1I=,9,churn,0.062701,122.889533,0.0,NaN,NaN,149.0


In [8]:
# --- summary and save ---

out_path = SUBSAMPLE_DIR / "panel_user_month.csv"
panel.to_csv(out_path, index=False)

print(f"Saved panel -> {out_path}")
print(f"Rows: {len(panel):,}  |  Users: {panel.msno.nunique():,} (of {len(users):,} sampled)")
print(f"Mean months per user: {len(panel) / panel.msno.nunique():.1f}")
print("\nSubscription status:")
print(panel["subscription_status"].value_counts())

def active_after_churn(g: pd.DataFrame) -> bool:
    g = g.sort_values("user_tenure")
    churn_months = g.loc[g["subscription_status"] == "churn", "user_tenure"]
    if churn_months.empty:
        return False
    return ((g["subscription_status"] == "active") & (g["user_tenure"] > churn_months.min())).any()

Saved panel -> kkbox-subsample/panel_user_month.csv
Rows: 21,812  |  Users: 2,465 (of 5,000 sampled)
Mean months per user: 8.8

Subscription status:
subscription_status
active    20899
churn       662
paused      251
Name: count, dtype: int64
